# NB00c – Review-Protokoll & Qualitätssicherung
### CAS Information Engineering – Scripting Project
**Gruppe:** SC26_Gruppe_2 | **Datum:** Mai 2026

---
## 1. Zweck dieses Dokuments
Dieses Projekt wurde unter Einsatz von Generativer KI (LLMs) als "Augmented Engineering"-Prozess umgesetzt. Um die akademische Integrität und die fachliche Korrektheit für den geschäftlichen Einsatz sicherzustellen, wurde ein strukturierter Review-Prozess implementiert.

Dieses Protokoll dokumentiert, dass die KI lediglich als **handwerkliches Werkzeug** (Code-Generierung) diente, während die **Architektur, Validierung und finale Entscheidungsgewalt** beim menschlichen Team lagen.

## 2. Rollenbasierte Qualitätssicherung

| Rolle | Fokus | Verantwortlich |
| :--- | :--- | :--- |
| **Domain Expert** | Physikalische Plausibilität, Netzlogik (Swissgrid/ENTSO-E), regulatorische Rahmenbedingungen CH. | [Dein Name] |
| **Financial Reviewer** | Validierung der ROI-Modelle, CAPEX-Lernkurven, Cash-Flow-Berechnungen (Banken-Standard). | [Name Kollege 2] |
| **System Architect** | Datenfluss-Design, `config.json` Konsistenz, API-Integration. | [Dein Name] |

## 3. Protokollierte Korrektur-Iterationen (Beispiele)

Die folgende Tabelle zeigt beispielhaft, wo der initiale KI-Output fachlich unzureichend war und durch das Team korrigiert werden musste:

| Modul | Initialer KI-Vorschlag | Menschlicher Review / Korrektur | Grund der Korrektur |
| :--- | :--- | :--- | :--- |
| **NB02 (Dispatch)** | Einfacher gleitender Durchschnitt für Lade-Entscheidungen. | Umstellung auf **p25/p75 Perzentil-Strategie** basierend auf Day-Ahead-Fixpreisen. | Reaktive Modelle unterschätzen das Arbitrage-Potenzial bei bekannten Preisen. |
| **NB01 (API)** | Statische URL-Struktur für ENTSO-E. | Implementierung einer dynamischen Datumssteuerung via `config.json` & Error Handling. | API-Anforderungen änderten sich; Robustheit für Langzeit-Analyse war nicht gegeben. |
| **NB14 (Produkt)** | Durchschnittliche CAPEX-Werte aus der Weltmarkt-Statistik. | Manuelle Korrektur auf **reale Schweizer Produktwerte** (7kW/14kWh Steckbrief). | Weltmarktpreise bilden die Schweizer Installations- und Importkosten nicht präzise ab. |
| **NB07 (Cross-Border)** | Korrelation von Preis und Volumen ohne Richtungsprüfung. | Trennung in **Import- vs. Export-Stunden** zur Validierung der Netzentlastung. | Nur die Richtungsanalyse beweist den systemischen Nutzen für das Schweizer Netz. |

## 4. Verifikation durch Code
Jedes Modul enthält am Ende jeder Sektion Verifikations-Zellen. Diese dienen nicht der Datenanalyse, sondern der **menschlichen Kontrolle des KI-Outputs**.

**Checkliste für jedes Notebook:**
- [x] Lädt Daten aus `config.json` (keine Hardcodierung)
- [x] Verifikations-Plots (Plotted die KI das, was wir erwarten?)
- [x] Unit-Tests / Assertions (Stimmen die Summen/Shapes?)

In [ ]:
import json
import os

# Pfad zur Config
CONFIG_FILE = 'config.json'

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        cfg = json.load(f)

    print("--- MANUELLER PLAUSIBILITÄTS-CHECK ---")
    
    try:
        # Die Struktur laut deiner Fehlermeldung: 
        # 'szenarien' -> 'wirtschaftlichkeit'
        if 'szenarien' in cfg and 'wirtschaftlichkeit' in cfg['szenarien']:
            wirth = cfg['szenarien']['wirtschaftlichkeit']
        # Falls es doch in 'pflicht' verschoben wurde:
        elif 'pflicht' in cfg and 'wirtschaftlichkeit' in cfg['pflicht']:
            wirth = cfg['pflicht']['wirtschaftlichkeit']
        else:
            raise KeyError("Sektion 'wirtschaftlichkeit' weder in 'szenarien' noch in 'pflicht' gefunden.")
            
        # Extraktion der Werte
        capex_privat = wirth['capex_eur_kwh']['Privat_10kWh']
        opex = wirth['opex_rate']
        lifetime = wirth['lifetime_j']
        eur_chf = cfg.get('eur_chf', 0.97)
        
        print(f"Geprüftes Segment : Privat_10kWh")
        print(f"Eingestellter CAPEX: {capex_privat} EUR/kWh")
        print(f"Eingestellter OPEX : {opex*100}% p.a.")
        print(f"Lebensdauer        : {lifetime} Jahre")
        print(f"Wechselkurs (Fix)  : {eur_chf} EUR/CHF")
        
        # Plausibilitäts-Urteil (deine menschliche Einschätzung)
        print(f"\n[MENSCHLICHER REVIEW-STATUS]")
        if capex_privat == 400:
            print("Status: VALIDATED (Standard-Szenario)")
        elif capex_privat < 200:
            print("Status: OPTIMISTISCH (Entspricht Zielwerten für 2026/2030)")
        else:
            print("Status: CUSTOM (Manuell angepasste Marktwerte)")
            
    except KeyError as e:
        print(f"Strukturfehler in config.json: {e}")
        # Kleiner Helfer, um die Struktur im Notfall zu sehen:
        if 'szenarien' in cfg:
            print(f"Unter-Keys in 'szenarien': {list(cfg['szenarien'].keys())}")
else:
    print(f"Fehler: {CONFIG_FILE} nicht gefunden.")